# AI工学101 — 第20回

## 汎化・過学習・正則化：なぜ「訓練データで100点」が危険なのか

よし、第20回まで来たぞ。
ここはかなり大事な節目だね。

第19回では、

```text
モデル
↓
交差検証
↓
ハイパーパラメータ探索
```

まで進みました。

でも、ここで一つ根本的な疑問が出ます。

> **そもそも、なぜ「学習データに完璧に合うモデル」が良いモデルとは限らないのか？**

今日のテーマは、機械学習のど真ん中にある

* **汎化（generalization）**
* **過学習（overfitting）**
* **未学習（underfitting）**
* **正則化（regularization）**

です。

ここを理解すると、今までやってきた `train/test`、Cross Validation、`C` の意味が一本につながります。

---

# 🎯 今日のゴール

今日の90分で、

* 過学習と未学習を説明できる
* train性能とtest性能の差を解釈できる
* 正則化が何をしているのか理解する
* Logistic Regressionの `C` と正則化の関係を理解する
* モデルの複雑さと汎化性能の関係を実験できる

ようにします。

---

# 📖 講義：約20〜25分

## 1. 機械学習の本当の目的

機械学習で本当に欲しいものは、

```text
訓練データを覚える能力
```

ではありません。

欲しいのは、

> **まだ見たことのないデータに対しても、うまく予測できる能力**

です。

これを

**汎化（Generalization）**

と呼びます。

---

## 2. 3つの状態

モデルの状態をざっくり3つに分けます。

### 未学習（Underfitting）

```text
モデルが単純すぎる
```

学習データに対しても十分に合わない。

```text
train性能：低い
test性能：低い
```

---

### ちょうどよい

```text
train性能：高い
test性能：高い
```

しかも両者の差が小さい。

これが理想。

---

### 過学習（Overfitting）

モデルが訓練データに合わせすぎる。

```text
train性能：非常に高い

test性能：低い
```

つまり、

> **訓練データでは天才なのに、初見のデータで急にポンコツになる**

状態です。

AI工学あるあるです。😂

---

# 🧠 3. なぜ過学習が起こる？

例えば、

```text
データが100個
```

しかないのに、

```text
モデルの自由度がものすごく高い
```

とします。

するとモデルは、

```text
本当に意味のある規則
```

だけでなく、

```text
偶然のノイズ
```

まで覚えてしまいます。

---

## 「規則」と「ノイズ」

例えば、

```text
身長 → 体重
```

にはある程度関係があります。

しかし、

```text
今日の気温 → この人の体重
```

のような偶然の相関を、モデルが「重要な法則」だと思ってしまう可能性があります。

これが過学習の一つのイメージです。

---

# 💻 実習1：過学習を目で見る

今回は人工データを作ります。

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

X = np.linspace(0, 10, 30)

y = (
    2 * X
    + 3
    + np.random.normal(0, 3, size=30)
)
```

確認。

```python
plt.scatter(X, y)
plt.show()
```

だいたい、

```text
右肩上がり
```

のデータになります。

---

# 💻 実習2：線形モデル

まず普通の線形回帰。

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(
    X.reshape(-1, 1),
    y
)
```

予測。

```python
pred = model.predict(
    X.reshape(-1, 1)
)
```

描画。

```python
plt.scatter(X, y)

plt.plot(
    X,
    pred
)

plt.show()
```

かなり自然な直線になります。

---

# 💻 実習3：多項式を強くする

第17回で使った

```python
PolynomialFeatures
```

を使います。

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_model = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=10,
            include_bias=False
        )
    ),
    (
        "model",
        LinearRegression()
    )
])
```

学習。

```python
poly_model.fit(
    X.reshape(-1, 1),
    y
)
```

予測。

```python
pred_poly = poly_model.predict(
    X.reshape(-1, 1)
)
```

描画。

```python
plt.scatter(X, y)

plt.plot(
    X,
    pred_poly
)

plt.show()
```

さて、どうなるでしょう。

---

## 👀 観察ポイント

degree=10では、

```text
データ点の間を激しく上下する
```

ような曲線になることがあります。

これは、

> **データのノイズまで拾っている**

状態です。

---

# 💻 実習4：train/test性能を比較する

データを分けます。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X.reshape(-1, 1),
    y,
    test_size=0.3,
    random_state=42
)
```

degreeを変えながら、

```python
degrees = [1, 2, 5, 10, 15]
```

それぞれモデルを作ってみます。

```python
from sklearn.metrics import mean_squared_error

for degree in degrees:

    pipe = Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=degree,
                include_bias=False
            )
        ),
        (
            "model",
            LinearRegression()
        )
    ])

    pipe.fit(
        X_train,
        y_train
    )

    train_pred = pipe.predict(
        X_train
    )

    test_pred = pipe.predict(
        X_test
    )

    train_mse = mean_squared_error(
        y_train,
        train_pred
    )

    test_mse = mean_squared_error(
        y_test,
        test_pred
    )

    print(
        degree,
        train_mse,
        test_mse
    )
```

---

# 🧠 ここで見るもの

例えば概念的には、

| degree | Train MSE | Test MSE |
| -----: | --------: | -------: |
|      1 |        高い |       高い |
|      2 |         ↓ |        ↓ |
|      5 |        低い |       低い |
|     10 |     とても低い |       上昇 |
|     15 |       ほぼ0 |    非常に高い |

という現象が起こり得ます。

つまり、

```text
モデルの複雑さ
       ↓
あるところまでは性能向上
       ↓
複雑にしすぎる
       ↓
過学習
```

です。

---

# 📖 5. 正則化とは？

では、モデルが複雑になりすぎるのをどう防ぐか。

そこで登場するのが

> **正則化（Regularization）**

です。

基本思想は、

> **「データに合っているだけではなく、モデルが必要以上に複雑にならないようにしよう」**

です。

---

## 直感的には

普通の学習では、

```text
予測誤差を小さくする
```

ことだけを考えます。

正則化では、

```text
予測誤差
+
複雑さへのペナルティ
```

を考えます。

つまり、

[
Loss
====

Data\ Error
+
Regularization
]

という発想です。

---

# 🧠 6. Logistic Regressionと正則化

ここで第19回に戻ります。

```python
LogisticRegression(C=1)
```

の `C`。

これは単なる謎の数字ではありません。

`C` は、

> **正則化の強さの逆方向**

として働きます。

ざっくり、

```text
C 小
↓
正則化 強い
↓
モデルを強く抑える
```

一方、

```text
C 大
↓
正則化 弱い
↓
モデルがより自由になる
```

です。

---

# 💻 実習5：Cを変えてみる

Irisを使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

分割。

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

## Cを変える

```python
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

C_values = [
    0.001,
    0.01,
    0.1,
    1,
    10,
    100
]
```

ループ。

```python
for C in C_values:

    pipe = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                C=C,
                max_iter=1000
            )
        )
    ])

    pipe.fit(
        X_train,
        y_train
    )

    train_acc = pipe.score(
        X_train,
        y_train
    )

    test_acc = pipe.score(
        X_test,
        y_test
    )

    print(
        "C =", C,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 👀 観察する

ここで、

```text
C

train accuracy

test accuracy
```

の関係を見ます。

ポイントは、

> **trainが一番高いCを選ぶことが目的ではない**

ということ。

最終的に欲しいのは、

**未知データへの性能**

だからです。

---

# 📖 7. Bias–Varianceを直感で理解する

今日は数式を深追いしません。

まずイメージだけ。

### Biasが大きい

```text
モデルが単純すぎる
```

→ 未学習しやすい。

### Varianceが大きい

```text
データに敏感すぎる
```

→ 過学習しやすい。

つまり、

```text
単純すぎる
    ↓
Underfitting

ちょうどいい
    ↓
Generalization

複雑すぎる
    ↓
Overfitting
```

というバランスがあります。

これを理解しておくと、

「なんでモデルを複雑にすればするほど良いわけじゃないの？」

という疑問がかなり整理されます。

---

# ✍️ 演習

## 問1

次の文章を自分の言葉で説明してください。

> 「train accuracyが100%なのに、test accuracyが70%」

これは何が起きている可能性が高いでしょう？

---

## 問2

次の3つを分類してください。

### A

```text
train accuracy = 60%
test accuracy = 58%
```

### B

```text
train accuracy = 99%
test accuracy = 70%
```

### C

```text
train accuracy = 94%
test accuracy = 92%
```

それぞれ、

```text
未学習

過学習

比較的良好な汎化
```

のどれに近いでしょう？

---

## 問3

`C` を小さくすると、

```text
正則化
```

は強くなるか、弱くなるか。

理由も説明してください。

---

## 問4

第17回の `PolynomialFeatures` に戻ります。

次のモデルについて考えてください。

```python
degree=1
degree=2
degree=10
degree=20
```

一般にdegreeを上げすぎると、どんな問題が起こり得るでしょう？

---

# 👾 ボス戦

ここからが今日の本丸。

次のコードを完成させてください。

```python
degrees = [1, 2, 3, 5, 10, 15]

for degree in degrees:

    model = Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=degree,
                include_bias=False
            )
        ),
        (
            "model",
            LinearRegression()
        )
    ])

    model.fit(
        X_train,
        y_train
    )

    train_pred = ...

    test_pred = ...

    train_mse = ...

    test_mse = ...

    print(
        degree,
        train_mse,
        test_mse
    )
```

そして結果を見て、

> **「どのdegreeが汎化性能という観点で良さそうか」**

を判断してください。

ここでは、

**train MSEが最小のdegreeを選ばない**

ことがポイントです。

---

# 🌱 今日のまとめ

第20回で、ここまでの内容がかなりきれいにつながりました。

```text
データ
 ↓
train / test
 ↓
モデル
 ↓
fit
 ↓
train性能
 ↓
test性能
 ↓
汎化性能
```

さらに、

```text
モデルが単純すぎる
        ↓
   Underfitting

モデルが適切
        ↓
 Generalization

モデルが複雑すぎる
        ↓
   Overfitting
```

そして、

```text
過学習を抑える
        ↓
正則化
```

です。

さらに第19回の `C` が、

```text
C
↓
正則化の強さ
```

とつながりました。

---

## 🧩 20回分の現在地

ここまでで、scikit-learnのかなり重要な骨格ができています。

```text
NumPy
  ↓
データ操作
  ↓
前処理
  ↓
特徴量
  ↓
回帰
  ↓
分類
  ↓
評価指標
  ↓
Pipeline
  ↓
Cross Validation
  ↓
Grid Search
  ↓
汎化
  ↓
過学習
  ↓
正則化
```

ここまで来ると、単に「scikit-learnの関数を覚えている」のではなく、

**機械学習実験を設計するための基本概念**

がかなり揃ってきています。

---

# 🔜 第21回

## 決定木：線形モデルでは表現できない「条件分岐」を学習する

次回はいったん線形モデルから離れます。

今までのモデルは、

```text
Wx + b
```

を基本としていました。

次は、

```text
もし x < 3 なら……
そうでなければ……
```

という**条件分岐そのものをデータから学習するモデル**、

**Decision Tree（決定木）**

に入ります。

そして決定木は、その先の

```text
Random Forest
Gradient Boosting
```

などのアンサンブル学習へつながっていきます。

ここから「機械学習モデルの種類を増やしながら、同じ評価・実験フレームワークで比較する」という、より実践的なフェーズに入るぞ。